# Testing Serialization Issues

In [1]:
import tensorflow as tf
import keras
import numpy as np
from objects.autoencoder import SparseAutoEncoder

2024-11-18 10:55:06.656997: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-18 10:55:06.665453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-11-18 10:55:06.676111: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-11-18 10:55:06.679195: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-18 10:55:06.687317: I tensorflow/core/platform/cpu_feature_guar

In [2]:
# create dataset from random tensors to test
SAE_name = 'autoencoder_test'
embed_length = 2048
ef = 4

print("=== Generating Test Data ===")
fake_embeddings = tf.random.uniform(shape=[10000, embed_length])
fake_dataset = tf.data.Dataset.from_tensor_slices((fake_embeddings, fake_embeddings)).batch(100)

print("=== Initializing Model ===")
# initialize the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

# initialize the loss function
loss = keras.losses.MeanSquaredError()

# initialize the metrics
metrics = [
    keras.metrics.MeanSquaredError(),
    keras.metrics.Metric(name='placeholder') # placeholder for training, feature output requires a 2nd metric to appease keras
]

# compile the model
autoencoder = SparseAutoEncoder(encoding_size=embed_length, expansion_factor=ef)
autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

tb_callback = keras.callbacks.TensorBoard(log_dir=f'./logs/{SAE_name}', histogram_freq=5)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='mean_squared_error',
    min_delta=0.001,
    patience=10, 
    restore_best_weights=True
    )

print("=== Training Model ===")
history = autoencoder.fit(fake_dataset, epochs=100, callbacks=[tb_callback, early_stopping])
print("=== Saving Model ===")
path = f'./models/{SAE_name}.keras'
autoencoder.save(path)
print(f"Model saved to: {path}")
print(autoencoder.summary())

=== Generating Test Data ===
=== Initializing Model ===
=== Training Model ===
Epoch 1/100


I0000 00:00:1731948907.943008   32988 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731948907.972402   32988 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731948907.972578   32988 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731948907.974173   32988 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

 13/100 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2238.9285 - mean_squared_error: 0.3142 

I0000 00:00:1731948910.725383   33079 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step - loss: 1440.6052 - mean_squared_error: 0.3224
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 327.3132 - mean_squared_error: 0.2339
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 354.3241 - mean_squared_error: 0.1539
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 341.8754 - mean_squared_error: 0.1212
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 333.5666 - mean_squared_error: 0.1891
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - loss: 335.7838 - mean_squared_error: 0.2016
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 332.7849 - mean_squared_error: 0.2147
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 338.9256 - mean_squared_error: 0.1755
Epoch 9/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 344.5506 - mean_squared_error: 0.2086
Epoch 10/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 340.4494 - mean_squared_error: 0.1689
Epoch 11/100


...

 Total params: 100,663,298 (384.00 MB)

 Trainable params: 33,554,432 (128.00 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 67,108,866 (256.00 MB)

None


In [3]:
autoencoder.get_config()

{'name': 'sparse_auto_encoder',
 'trainable': True,
 'dtype': {'module': 'keras',
  'class_name': 'DTypePolicy',
  'config': {'name': 'float32'},
  'registered_name': None},
 'encoding_size': 8192,
 'expansion_factor': 1}

In [4]:
for x in autoencoder.get_weights():
    print(x.shape)

(2048, 8192)
(8192, 2048)


In [5]:
autoencoder.optimizer.get_config()

{'name': 'adam',
 'learning_rate': 0.0010000000474974513,
 'weight_decay': None,
 'clipnorm': None,
 'global_clipnorm': None,
 'clipvalue': None,
 'use_ema': False,
 'ema_momentum': 0.99,
 'ema_overwrite_frequency': None,
 'loss_scale_factor': None,
 'gradient_accumulation_steps': None,
 'beta_1': 0.9,
 'beta_2': 0.98,
 'epsilon': 1e-09,
 'amsgrad': False}

In [6]:
model = keras.models.load_model('./models/autoencoder_test.keras')

ValueError: A total of 2 objects could not be loaded. Example error message for object <Dense name=decoder, built=True>:

Layer 'decoder' expected 1 variables, but received 0 variables during loading. Expected: ['kernel']

List of objects that could not be loaded:
[<Dense name=decoder, built=True>, <keras.src.optimizers.adam.Adam object at 0x7cbed256a240>]

### Workaround for Model Saving / Loading
Github Issue 19615 in Keras Thread with reports of persistence:
https://github.com/keras-team/keras/issues/19615

In [7]:
autoencoder.save_weights('./models/autoencoder_test.weights.h5')

In [8]:
# Clear all previously registered custom objects
keras.saving.get_custom_objects().clear()

new_model = SparseAutoEncoder(encoding_size=2048, expansion_factor=4)
new_model.load_weights('./models/autoencoder_test.weights.h5')

In [11]:
new_model(fake_embeddings)

(<tf.Tensor: shape=(10000, 2048), dtype=float32, numpy=
 array([[0.        , 0.        , 0.15827896, ..., 0.06191964, 0.70137477,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.28178042, 0.46300757,
         0.        ],
        [0.01733182, 0.        , 0.01922595, ..., 0.07758948, 0.40666208,
         0.        ],
        ...,
        [0.        , 0.        , 0.        , ..., 0.18155588, 0.39852828,
         0.        ],
        [0.03395806, 0.        , 0.        , ..., 0.04420577, 0.5134706 ,
         0.        ],
        [0.        , 0.        , 0.        , ..., 0.3234222 , 0.67129755,
         0.        ]], dtype=float32)>,
 <tf.Tensor: shape=(10000, 8192), dtype=float32, numpy=
 array([[0.        , 0.        , 0.        , ..., 0.2841797 , 0.7343886 ,
         0.06922611],
        [0.        , 0.22098026, 0.        , ..., 0.57323873, 0.6862416 ,
         0.10300922],
        [0.        , 0.13380192, 0.43372986, ..., 0.33949393, 0.6185459 ,
         0.4029